相似性搜索:
- 最大边际相关MMR
- 元数据

In [2]:
### 加载之前已经保存的向量数据库

from langchain_community.vectorstores import Chroma
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

persist_directory = 'data/matplotlib/'

embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

embedding_chinese = HuggingFaceEmbeddings(
    model_name="shibing624/text2vec-base-chinese",
    model_kwargs={"device": "cpu"}
)

vectordb = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_chinese
)

print(vectordb._collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: shibing624/text2vec-base-chinese
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\61075\AppData\Local\Temp\ipykernel_22260\4275726737.py:18: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(


49


Maximal Marginal Relevance：解决多样性

In [5]:
### 最大边际相关模型，同时考虑查询和文档的相关性，以及文档之间的相似度

# 创建一个小db
texts = [
    """The Amanita phalloides has a large and imposing epigeous (aboveground) fruiting body (basidiocarp).""",
    """A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.""",
    """A. phalloides, a.k.a Death Cap, is one of the most poisonous of all known mushrooms.""",
]

smalldb = Chroma.from_texts(
    texts=texts,
    embedding=embedding
)

question = "Tell me about all-white mushrooms with large fruiting bodies"

print("similarity search: \n", smalldb.similarity_search(question, k=2))

print("maximal marginal relevance: \n", smalldb.max_marginal_relevance_search(question, k=2, fetch_k=3))

similarity search: 
 [Document(metadata={}, page_content='A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.'), Document(metadata={}, page_content='A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.')]
maximal marginal relevance: 
 [Document(metadata={}, page_content='A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.'), Document(metadata={}, page_content='A mushroom with a large fruiting body is the Amanita phalloides. Some varieties are all-white.')]


In [8]:
question_chinese = "Matplotlib是什么？"
docs_ss_chinese = vectordb.similarity_search(question_chinese,k=3)

print("docs[0]: ")
print(docs_ss_chinese[0].page_content[:100])
print()
print("docs[1]: ")
print(docs_ss_chinese[1].page_content[:100])

docs[0]: 
第⼆回：艺术画笔⻅乾坤
import numpy as np
import pandas as pd
import re
import matplotlib
import matplotlib.pyp

docs[1]: 
第⼀回： Matplotlib 初相识
⼀、认识 matplotlib
Matplotlib 是⼀个 Python 2D 绘图库，能够以多种硬拷⻉格式和跨平台的交互式环境⽣成出版物质量的图形，⽤来
绘


In [9]:
docs_mmr_chinese = vectordb.max_marginal_relevance_search(question_chinese,k=3)

print("docs[0]: ")
print(docs_mmr_chinese[0].page_content[:100])
print()
print("docs[1]: ")
print(docs_mmr_chinese[1].page_content[:100])

docs[0]: 
第⼆回：艺术画笔⻅乾坤
import numpy as np
import pandas as pd
import re
import matplotlib
import matplotlib.pyp

docs[1]: 
第⼀回： Matplotlib 初相识
⼀、认识 matplotlib
Matplotlib 是⼀个 Python 2D 绘图库，能够以多种硬拷⻉格式和跨平台的交互式环境⽣成出版物质量的图形，⽤来
绘


metadata：使用元数据过滤

In [23]:
question_chinese = "他们在第二讲中对Figure说了些什么？"

# 指定元数据过滤器，手动设置过滤条件
docs_chinese = vectordb.similarity_search(
    question_chinese,
    k=3,
    filter={"source":"data/matplotlib/第二回：艺术画笔见乾坤 — fantastic-matplotlib.pdf"}
)

for d in docs_chinese:
    print(d.metadata)
    print()

{'total_pages': 15, 'moddate': '2026-03-17T09:45:57+00:00', 'creationdate': '2026-03-17T09:45:57+00:00', 'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36', 'source': 'data/matplotlib/第二回：艺术画笔见乾坤 — fantastic-matplotlib.pdf', 'title': '第二回：艺术画笔见乾坤 — fantastic-matplotlib', 'page': 11, 'page_label': '12'}

{'total_pages': 15, 'producer': 'Skia/PDF m145', 'page_label': '11', 'creationdate': '2026-03-17T09:45:57+00:00', 'source': 'data/matplotlib/第二回：艺术画笔见乾坤 — fantastic-matplotlib.pdf', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36', 'moddate': '2026-03-17T09:45:57+00:00', 'title': '第二回：艺术画笔见乾坤 — fantastic-matplotlib', 'page': 10}

{'source': 'data/matplotlib/第二回：艺术画笔见乾坤 — fantastic-matplotlib.pdf', 'moddate': '2026-03-17T09:45:57+00:00', 'page': 0, 'total_pages': 15, 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win

In [1]:
## 元数据中使用自查询检索器SelfQueryRetriever（LLM辅助检索）

from langchain_ollama import ChatOllama
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.base import AttributeInfo

llm = ChatOllama(model='qwen2.5:7b' ,temperature=0.0)

In [7]:
# 定义metadata_field_info，
# 包含元数据的过滤条件“source”（告诉LLM想要的数据来源）
# 和“page”（告诉LLM需要提取相关的内容在原文档的哪一页）

metadata_field_info = [
    AttributeInfo(
        name='source',
        description="The lecture the chunk is from, should be one of `data/matplotlib/第一回：Matplotlib初相识 — fantastic-matplotlib.pdf`, `data/matplotlib/第二回：艺术画笔见乾坤 — fantastic-matplotlib.pdf`, or `data/matplotlib/第三回：布局格式定方圆 — fantastic-matplotlib.pdf`",
        type="string"
    ),
    AttributeInfo(
        name='page',
        description="文档页码，用于定位内容位置",
        type="integer"
    )
]


# SelfQueryRetriever模块，使用LLM在query中分析出向量搜索的查询字符串和过滤文档的元数据条件
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectordb,
    document_contents="matplotlib 课堂讲义",
    metadata_field_info=metadata_field_info,
    verbose=1,
)



In [15]:
question = "第五回中提到了什么绘图样式？"

docs = retriever.invoke(question)
for doc in docs:
    # print(doc.page_content)
    print(doc.metadata)
    print()


{'title': '第三回：布局格式定方圆 — fantastic-matplotlib', 'creationdate': '2026-03-17T09:46:07+00:00', 'total_pages': 5, 'page': 0, 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36', 'producer': 'Skia/PDF m145', 'moddate': '2026-03-17T09:46:07+00:00', 'source': 'data/matplotlib/第三回：布局格式定方圆 — fantastic-matplotlib.pdf', 'page_label': '1'}

{'source': 'data/matplotlib/第三回：布局格式定方圆 — fantastic-matplotlib.pdf', 'page_label': '3', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36', 'total_pages': 5, 'page': 2, 'title': '第三回：布局格式定方圆 — fantastic-matplotlib', 'moddate': '2026-03-17T09:46:07+00:00', 'creationdate': '2026-03-17T09:46:07+00:00', 'producer': 'Skia/PDF m145'}

{'moddate': '2026-03-17T09:46:07+00:00', 'creationdate': '2026-03-17T09:46:07+00:00', 'producer': 'Skia/PDF m145', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (K

Compression

In [14]:
### 压缩：直接返回整个文档可能带来资源浪费
### 先使用标准向量检索到候选文档，然后基于查询语句的语义，使用语言模型压缩这些文档，只保留与问题相关的部分
### 压缩其实就是从文档块中提取出和query相关的内容

from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

llm = ChatOllama(model='qwen2.5:7b' ,temperature=0.0)

# 压缩器
compressor = LLMChainExtractor.from_llm(llm=llm)

compression_retriever = ContextualCompressionRetriever(
    base_retriever=vectordb.as_retriever(),
    base_compressor=compressor,
)

# 对源文档进行压缩
question = "有哪些绘图样式"
compressed_docs = compression_retriever.invoke(question)
print(f"\n{'-' * 100}\n".join([f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(compressed_docs)]))

Document 1:

有哪些绘图样式，常⻅的有 3 种⽅法，分别是修改预定义样式，⾃定义样式和 rcparams 。
----------------------------------------------------------------------------------------------------
Document 2:

# step2 设置绘图样式，这一模块的扩展参考第五章进一步学习，这一步不是必须的，样式也可以在绘制图像  
mpl.rc('lines', linewidth=4, linestyle='-.')
